**LABORATORIO 5: Pytorch - Redes Neuronales**

ESTUDIANTE: DIAZ CAMPOS NATALIA CAMILA

CARRERA: INGENIERÍA DE SISTEMAS

# **PARTE 1: Carga y Exploración de Datos**

Basado en el Cuadernillo 2 (Datasets) y Cuadernillo 3 (Redes). El primer paso es cargar el archivo y verificar los nombres de las columnas. Nunca debemos asumir que se llaman igual que en otros ejemplos.

In [44]:
import torch
import pandas as pd
import numpy as np
from google.colab import files
import io

# Subida del archivo
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
data = pd.read_csv(io.BytesIO(uploaded[file_name]))

# EXPLORACIÓN: Esto es vital para evitar el KeyError
print("Columnas detectadas:", data.columns.tolist())
data.head()

Saving amazon_sales_dataset.csv to amazon_sales_dataset (9).csv
Columnas detectadas: ['order_id', 'order_date', 'product_id', 'product_category', 'price', 'discount_percent', 'quantity_sold', 'customer_region', 'payment_method', 'rating', 'review_count', 'discounted_price', 'total_revenue']


,order_id,order_date,product_id,product_category,price,discount_percent,quantity_sold,customer_region,payment_method,rating,review_count,discounted_price,total_revenue
0,1,2022-04-13,2637,Books,128.75,10,4,North America,UPI,3.5,443,115.88,463.52
1,2,2023-03-12,2300,Fashion,302.60,20,5,Asia,Credit Card,3.7,475,242.08,1210.40
2,3,2022-09-28,3670,Sports,495.80,20,2,Europe,UPI,4.4,183,396.64,793.28
3,4,2022-04-17,2522,Books,371.95,15,4,Middle East,UPI,5.0,212,316.16,1264.64
4,5,2022-03-13,1717,Beauty,201.68,0,4,Middle East,UPI,4.6,308,201.68,806.72


# **PARTE 2: Preprocesamiento y Limpieza**

Basado en el Cuadernillo 3 (Redes). Las redes neuronales solo entienden números. Aquí convertiremos las categorías a códigos y limpiaremos los valores que tengan símbolos extraños.

Nota: He ajustado los nombres a price y discounted_price que son los que tiene tu dataset de Ali Hussain.

In [45]:
# 1. Definir el Target (Lo que queremos predecir)
# Usaremos 'product_category' como en tu ejemplo original
data["Target"] = data["product_category"].astype("category").cat.codes
n_clases = len(data["Target"].unique())

# 2. Limpieza de columnas numéricas
# Si el precio viene como texto (ej. "₹500"), lo limpiamos.
# Si ya es número, el código lo ignorará sin error.
def clean_num(col):
    if data[col].dtype == 'object':
        return data[col].str.replace('₹', '').str.replace(',', '').astype(float)
    return data[col]

data['price'] = clean_num('price')
data['discounted_price'] = clean_num('discounted_price')
data['rating'] = pd.to_numeric(data['rating'], errors='coerce').fillna(0)

# 3. Selección de características (X) y Etiquetas (Y)
# Usamos las columnas numéricas para entrenar
features = ['price', 'discounted_price', 'discount_percent', 'quantity_sold', 'rating']
X = data[features].values
Y = data["Target"].values

# 4. Normalización (Restar media y dividir por desviación estándar)
def featureNormalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

X_norm, mu, sigma = featureNormalize(X)

# 5. División: Entrenamiento (80%) y Prueba (20%)
split = int(0.8 * len(X_norm))
X_train, X_test = X_norm[:split], X_norm[split:]
y_train, y_test = Y[:split], Y[split:]

print(f"Entradas: {X.shape[1]} | Clases a predecir: {n_clases}")

Entradas: 5 | Clases a predecir: 6


# **PARTE 3: La Clase Dataset y DataLoader**

Basado en el Cuadernillo 2 (Datasets). Para que PyTorch trabaje de forma eficiente, encapsulamos los datos en una clase que hereda de Dataset. Esto permite usar DataLoader para entrenar por batches (lotes).

In [46]:
# 3.1. Definir el dispositivo (CPU o GPU) automáticamente
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Trabajando con: {device}")

# 3.2. Clase Dataset (Siguiendo el Cuadernillo 2)
class AmazonDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y):
        # Convertimos a tensores en CPU primero (más seguro)
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).long()

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, ix):
        # Retornamos los datos (el DataLoader se encargará de moverlos si es necesario)
        return self.X[ix], self.Y[ix]

# 3.3. Instanciación de los cargadores
train_ds = AmazonDataset(X_train, y_train)
test_ds = AmazonDataset(X_test, y_test)

# Usamos batch_size=100 y shuffle=True para que el modelo no aprenda el orden de los datos
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=100, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=100, shuffle=False)

print("DataLoader configurado con éxito y listo para cualquier entorno.")

Trabajando con: cpu
DataLoader configurado con éxito y listo para cualquier entorno.


# **PARTE 4: Modelos (Secuencial y Custom)**

Basado en el Cuadernillo 3 (Redes). Aquí definimos la arquitectura de la red. Usaremos las dimensiones dinámicas (D_in y D_out) que calculamos antes.

**4.1 Modelo Secuencial (Fácil)**

In [47]:
# 4.1. Definición de dimensiones usando las variables en memoria
D_in = X_train.shape[1]      # Entradas (columnas de X)
H = 100                      # Neuronas en la capa oculta
D_out = len(np.unique(Y))    # Usamos Y (mayúscula) para las categorías

# Verificamos dispositivo (CPU o GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 4.2. Modelo Secuencial (Cuadernillo 3)
model_seq = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out)
).to(device)

# 4.3. Modelo Custom con Conexión Residual (Cuadernillo 3)
class ModelCustomResidual(torch.nn.Module):
    def __init__(self, D_in, H, D_out):
        super(ModelCustomResidual, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        # Flujo con conexión residual
        x1 = self.fc1(x)
        x = self.relu(x1)
        # Sumamos la identidad (x1) para mejorar el flujo del gradiente
        x = self.fc2(x + x1)
        return x

# Instanciamos el modelo principal
model = ModelCustomResidual(D_in, H, D_out).to(device)

print(f"Dimensiones configuradas: Entrada={D_in}, Salida={D_out}")
print(f"Modelo cargado correctamente en: {device}")

Dimensiones configuradas: Entrada=5, Salida=6
Modelo cargado correctamente en: cpu


# **PARTE 5: Entrenamiento y Guardado**
Basado en el Cuadernillo 1 (Guardado). Implementaremos el bucle de entrenamiento que guarda el mejor checkpoint automáticamente.

In [53]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01) # Bajamos el lr para estabilidad

epochs = 500
PATH = './best_checkpoint.pt'
best_loss = float('inf')

model.train()
for e in range(1, epochs + 1):
    losses = []
    for x_b, y_b in train_loader:
        # Forward
        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    avg_loss = np.mean(losses)

    # GUARDAR MODELO: Si el error baja, guardamos este estado (Cuadernillo 1)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), PATH)

    if e % 10 == 0:
        print(f"Epoch {e}/{epochs} - Loss: {avg_loss:.5f}")

print("Entrenamiento completado y mejor modelo guardado.")

Epoch 10/500 - Loss: 1.79090
Epoch 20/500 - Loss: 1.79088
Epoch 30/500 - Loss: 1.79085
Epoch 40/500 - Loss: 1.79084
Epoch 50/500 - Loss: 1.79082
Epoch 60/500 - Loss: 1.79080
Epoch 70/500 - Loss: 1.79078
Epoch 80/500 - Loss: 1.79077
Epoch 90/500 - Loss: 1.79075
Epoch 100/500 - Loss: 1.79073
Epoch 110/500 - Loss: 1.79072
Epoch 120/500 - Loss: 1.79071
Epoch 130/500 - Loss: 1.79070
Epoch 140/500 - Loss: 1.79068
Epoch 150/500 - Loss: 1.79067
Epoch 160/500 - Loss: 1.79066
Epoch 170/500 - Loss: 1.79065
Epoch 180/500 - Loss: 1.79064
Epoch 190/500 - Loss: 1.79063
Epoch 200/500 - Loss: 1.79061
Epoch 210/500 - Loss: 1.79060
Epoch 220/500 - Loss: 1.79059
Epoch 230/500 - Loss: 1.79057
Epoch 240/500 - Loss: 1.79058
Epoch 250/500 - Loss: 1.79055
Epoch 260/500 - Loss: 1.79055
Epoch 270/500 - Loss: 1.79053
Epoch 280/500 - Loss: 1.79052
Epoch 290/500 - Loss: 1.79052
Epoch 300/500 - Loss: 1.79050
Epoch 310/500 - Loss: 1.79050
Epoch 320/500 - Loss: 1.79049
Epoch 330/500 - Loss: 1.79046
Epoch 340/500 - Los

In [51]:
from sklearn.metrics import accuracy_score

# 5.1. Cargar el mejor estado guardado (Cuadernillo 1)
# Usamos map_location para asegurar que cargue en CPU si no hay GPU
model.load_state_dict(torch.load(PATH, map_location=torch.device(device)))
model.eval()

# 5.2. Preparar datos de prueba en el dispositivo correcto
X_test_t = torch.from_numpy(X_test).float().to(device)

# 5.3. Realizar predicciones sin calcular gradientes (más rápido)
with torch.no_grad():
    y_pred_logits = model(X_test_t)
    # Obtenemos el índice de la probabilidad más alta
    y_pred_labels = torch.argmax(y_pred_logits, axis=1).cpu().numpy()

# 5.4. Calcular precisión final
# Comparamos Y (etiquetas reales) con y_pred_labels (predicciones)
acc = accuracy_score(y_test, y_pred_labels)
print(f"Precisión Final (Accuracy): {acc * 100:.2f}%")

Precisión Final (Accuracy): 16.74%


# **PARTE 6: Evaluación y Exportación a ONNX**
Basado en el Cuadernillo 1 (Exportando). Cargaremos el mejor modelo y lo exportaremos a formato ONNX para que sea "portable".

In [52]:
# 6.0. Instalación de la dependencia faltante
!pip install onnxscript

import torch.onnx

# 6.1. Crear el input de ejemplo
# D_in ya debe estar definido en tus celdas anteriores
dummy_input = torch.randn(1, D_in).to(device)

# 6.2. Exportar el modelo (Cuadernillo 1: Exportando modelos)
print("Exportando modelo a ONNX...")

torch.onnx.export(
    model,
    dummy_input,
    "amazon_model.onnx",
    verbose=False,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}},
    opset_version=14 # Usamos una versión estable de ONNX
)

print("¡Éxito! El archivo 'amazon_model.onnx' ya aparece en el panel izquierdo (carpeta).")

/tmp/ipykernel_848/2022080019.py:13: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0331 15:24:51.807000 848 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Exportando modelo a ONNX...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


¡Éxito! El archivo 'amazon_model.onnx' ya aparece en el panel izquierdo (carpeta).


### Probando con una tasa de aprendizaje más alta

Vamos a intentar entrenar el modelo con una tasa de aprendizaje (`lr`) de `0.01` en lugar de `0.001` para ver si mejora la convergencia.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer_new_lr = torch.optim.SGD(model.parameters(), lr=0.01) # Tasa de aprendizaje más alta

epochs = 500
PATH_new_lr = './best_checkpoint_new_lr.pt'
best_loss_new_lr = float('inf')

# Resetear el modelo a su estado inicial antes de re-entrenar,
# o cargar el estado inicial guardado si se desea una prueba "limpia"
# Por ahora, seguiremos entrenando sobre el estado actual, lo cual está bien para una primera prueba

model.train()
print(f"Entrenando con lr=0.01...")
for e in range(1, epochs + 1):
    losses = []
    for x_b, y_b in train_loader:
        # Forward
        y_pred = model(x_b)
        loss = criterion(y_pred, y_b)

        # Backward
        optimizer_new_lr.zero_grad()
        loss.backward()
        optimizer_new_lr.step()

        losses.append(loss.item())

    avg_loss = np.mean(losses)

    # GUARDAR MODELO: Si el error baja, guardamos este estado
    if avg_loss < best_loss_new_lr:
        best_loss_new_lr = avg_loss
        torch.save(model.state_dict(), PATH_new_lr)

    if e % 50 == 0: # Imprimir menos frecuentemente para no saturar la salida
        print(f"Epoch {e}/{epochs} - Loss: {avg_loss:.5f}")

print("Entrenamiento completado con nueva LR y mejor modelo guardado.")

### Evaluación con la nueva tasa de aprendizaje

In [ ]:
from sklearn.metrics import accuracy_score

# Cargar el mejor estado guardado con la nueva LR
model.load_state_dict(torch.load(PATH_new_lr, map_location=torch.device(device)))
model.eval()

# Preparar datos de prueba en el dispositivo correcto
X_test_t = torch.from_numpy(X_test).float().to(device)

# Realizar predicciones sin calcular gradientes
with torch.no_grad():
    y_pred_logits = model(X_test_t)
    y_pred_labels = torch.argmax(y_pred_logits, axis=1).cpu().numpy()

# Calcular precisión final
acc_new_lr = accuracy_score(y_test, y_pred_labels)
print(f"Precisión Final (Accuracy) con lr=0.01: {acc_new_lr * 100:.2f}%")